# Esegui i file src dal Notebook

Questo notebook clona il repository per ottenere il codice in `src/`, scarica i dati dal datastore `workspaceblobstore`, rende importabili i moduli e avvia train/evaluate.

> Il codice sorgente NON viene preso dallo storage: vive nel repository. I dati invece sono nel datastore.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/nicolgit/AMA14-assignment')
NOTEBOOK_DIR = Path.cwd().resolve()
REPO_DIR = NOTEBOOK_DIR / 'AMA14-assignment'
LOCAL_SRC_DIR = REPO_DIR / 'src'
LOCAL_DATA_DIR = NOTEBOOK_DIR / 'CMAPPS-data'
OUTPUT_MODEL_DIR = NOTEBOOK_DIR / 'outputs'
OUTPUT_EVAL_DIR = NOTEBOOK_DIR / 'eval'

for p in [LOCAL_DATA_DIR, OUTPUT_MODEL_DIR, OUTPUT_EVAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Notebook dir:', NOTEBOOK_DIR)
print('Repo dir:', REPO_DIR)
print('Local src dir:', LOCAL_SRC_DIR)

In [ ]:
def run(cmd: str) -> str:
    print('>>', cmd)
    out = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
    if out.stdout.strip():
        print(out.stdout.strip())
    if out.stderr.strip():
        print(out.stderr.strip())
    return out.stdout.strip()

# Verifica che Azure CLI sia disponibile e autenticata
run('az account show --output table')

In [ ]:
RG = os.environ.get('RG_NAME', 'ama-mro-playground')
acct = run(f"az storage account list -g {RG} --query \"[?starts_with(name,'stml')].name | [0]\" -o tsv")
mlw = run(f"az ml workspace list -g {RG} --query \"[0].name\" -o tsv")
container = run(f"az ml datastore show -g {RG} -w {mlw} -n workspaceblobstore --query container_name -o tsv")

print('Storage account:', acct)
print('ML workspace:', mlw)
print('Container:', container)

In [ ]:
# Ottieni il codice sorgente dal repository (non dallo storage)
if REPO_DIR.exists():
    run(f"git -C {REPO_DIR.as_posix()} pull")
else:
    run(f"git clone {REPO_URL} {REPO_DIR.as_posix()}")

# Scarica i dati raw dal datastore se non già presenti localmente
run(
    f"az storage blob download --account-name {acct} --auth-mode login -c {container} "
    f"-n raw/cmapss/fd004/train/train_FD004.txt -f {(LOCAL_DATA_DIR / 'train_FD004.txt').as_posix()} --overwrite"
)
run(
    f"az storage blob download --account-name {acct} --auth-mode login -c {container} "
    f"-n raw/cmapss/fd004/test/test_FD004.txt -f {(LOCAL_DATA_DIR / 'test_FD004.txt').as_posix()} --overwrite"
)

In [ ]:
# Rendi importabili i moduli scaricati
if str(LOCAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_SRC_DIR))

from preprocess import load_cmapss_file
from train import CNNLSTMRegressor

print('Import OK')
print('Rows in train file:', len(load_cmapss_file(str(LOCAL_DATA_DIR / 'train_FD004.txt'))))

In [ ]:
# Esegui training
run(
    f"python {LOCAL_SRC_DIR / 'train.py'} "
    f"--train_data {LOCAL_DATA_DIR / 'train_FD004.txt'} "
    f"--model_output {OUTPUT_MODEL_DIR} --epochs 5 --batch_size 128"
)

In [ ]:
# Esegui valutazione (senza label true in questa demo)
run(
    f"python {LOCAL_SRC_DIR / 'evaluate.py'} "
    f"--test_data {LOCAL_DATA_DIR / 'test_FD004.txt'} "
    f"--model_dir {OUTPUT_MODEL_DIR} --output_dir {OUTPUT_EVAL_DIR}"
)

print('Output predictions:', OUTPUT_EVAL_DIR / 'predictions.csv')
print('Output metrics:', OUTPUT_EVAL_DIR / 'evaluation.json')